# 3일차 실습 4 - /chat API 통합 구현

## 실습 목표

지금까지 따로 다뤘던 것들을 하나의 `/chat` 엔드포인트로 통합합니다.

- FastAPI 앱 + LLM 호출
- Request Model / Response Model 적용
- 기본 오류 처리 (try/except → HTTPException)
- `/docs`로 정상 / 실패 케이스 테스트

이번 실습이 끝나면 외부에 열어도 어색하지 않은 형태의 `/chat` API가 완성됩니다.

## 1. 라이브러리 불러오기

In [1]:
import os
from getpass import getpass
from dotenv import load_dotenv
from openai import OpenAI

## 2. LLM 호출 함수 정리

이전 실습에서 만든 LLM 호출 함수를 다시 정리합니다.

- API Key, Base URL, 모델명은 `.env` 파일에서 먼저 읽고,
  값이 없으면 입력 프롬프트로 직접 받습니다.
- LLM 클라이언트는 OpenAI 또는 교육 환경에서 제공되는 MLAPI 접속 정보를 사용합니다.

In [2]:
load_dotenv()

base_url   = os.getenv("MLAPI_BASE_URL", "https://mlapi.run/40cc17ae-a89b-4f12-a7d6-13293180fc87/v1")
api_key    = os.getenv("MLAPI_API_KEY")
model_name = os.getenv("MLAPI_MODEL", "openai/gpt-4o-mini")

if not api_key or api_key.startswith("여기에"):
    api_key = getpass("MLAPI API Key를 입력하세요: ")

client = OpenAI(base_url=base_url, api_key=api_key)

print("Base URL:", client.base_url)
print("모델명:", model_name)

Base URL: https://mlapi.run/8ebfcef5-8d34-4355-a8d9-89f9253acd2b/v1/
모델명: openai/gpt-5.4


이제 LLM 호출 함수를 정의합니다.

- 입력: 사용자 메시지(문자열), temperature(선택)
- 출력: LLM 응답 객체 전체 (`choices`, `usage` 등 포함)

In [3]:
def call_llm(message: str, temperature: float = 0.7):
    response = client.chat.completions.create(
        model=model_name,
        messages=[{"role": "user", "content": message}],
        temperature=temperature,
    )
    return response

In [5]:
# 정상 동작 확인
resp = call_llm("안녕? 너는 누구야?", temperature=0.5)
print(resp.choices[0].message.content)

안녕! 나는 OpenAI가 만든 AI 어시스턴트야.  
질문에 답하거나, 글쓰기/번역/요약/아이디어 정리/코딩 같은 걸 도와줄 수 있어.

편하게 말해줘. 뭐 도와줄까?


## 3. Request / Response Model 정의

`/chat` 엔드포인트가 받을 입력과 내려줄 출력을 Pydantic 모델로 먼저 정의합니다.

- `ChatRequest`: 클라이언트로부터 받는 입력 모델
  - `message`: 사용자 질문 (문자열, 필수)
- `ChatResponse`: 클라이언트에 내려줄 출력 모델
  - `reply`: LLM이 생성한 답변 텍스트
  - `model`: 어떤 모델이 응답했는지

In [6]:
from pydantic import BaseModel

class ChatRequest(BaseModel):
    message: str

class ChatResponse(BaseModel):
    reply: str
    model: str

## 4. /chat 엔드포인트 구현

3번에서 정의한 `ChatRequest`, `ChatResponse`를 적용해 `/chat` 엔드포인트를 만듭니다.

흐름:
1. 클라이언트가 보낸 JSON → `ChatRequest`로 자동 검증
2. `req.message`를 `call_llm`에 넘겨 LLM 호출
3. LLM 응답에서 필요한 값만 골라 `ChatResponse`로 정리해서 반환

In [7]:
from fastapi import FastAPI

app = FastAPI()

@app.post("/chat", response_model=ChatResponse)
def chat(req: ChatRequest):
    response = call_llm(req.message)
    reply_text = response.choices[0].message.content
    return ChatResponse(reply=reply_text, model=response.model)

포인트
- `response_model=ChatResponse`를 지정하면, 함수가 어떤 형태로 반환하든 결국 `ChatResponse` 형태로 정리되어 내려갑니다.
- LLM 응답에서 우리가 필요한 값은 `choices[0].message.content` 와 `response.model` 두 개입니다.
- 아직 오류 처리는 들어가지 않은 상태입니다. 다음 단계에서 붙입니다.

## 5. 오류 처리 붙이기

LLM API는 외부 서비스라 언제든 실패할 수 있습니다.
- API Key 만료, Rate limit, 네트워크 일시 단절, 모델명 오류 등

오류 처리의 기본 원칙:
- LLM 호출 한 줄만 try/except로 감싸기
- 잡은 오류는 `HTTPException`으로 변환해서 클라이언트에 내려주기
- 원본 예외 메시지를 그대로 노출하지 않기

In [8]:
from fastapi import HTTPException

app = FastAPI()

@app.post("/chat", response_model=ChatResponse)
def chat(req: ChatRequest):
    try:
        response = call_llm(req.message)
    except Exception:
        raise HTTPException(status_code=502, detail="LLM 호출 실패")
    reply_text = response.choices[0].message.content
    return ChatResponse(reply=reply_text, model=response.model)

포인트
- try 블록 안에는 **외부 호출 한 줄**만 들어갑니다. 우리 코드의 로직(`choices[0].message.content` 접근 등)은 try 바깥에 둡니다.
- 지금은 모든 실패를 일단 `502`로 내리지만, 이후 TODO에서 오류 유형별로 상태 코드를 분기합니다.
- `detail`에는 짧은 카테고리만 넣고, 원본 예외 메시지(`str(e)`)는 절대 그대로 노출하지 않습니다.

## 6. 서버 실행 및 /docs 테스트

작성한 FastAPI 앱을 노트북 안에서 직접 실행합니다.

- 노트북에서 uvicorn.run()을 그냥 호출하면 이벤트 루프와 충돌하므로, 별도 스레드에서 uvicorn을 실행합니다.
- 실행 후 출력되는 주소에 /docs를 붙여 접속하면 Swagger UI에서 바로 호출해볼 수 있습니다.
- 서버를 멈추려면 server.should_exit = True 를 실행하거나 커널을 재시작합니다.

In [9]:
import threading
import uvicorn
from uvicorn import Config, Server

config = Config(app=app, host="0.0.0.0", port=8000, log_level="info")
server = Server(config)

thread = threading.Thread(target=server.run, daemon=True)
thread.start()

print("서버 실행 중: http://localhost:8000")
print("문서 페이지: http://localhost:8000/docs")

서버 실행 중: http://localhost:8000
문서 페이지: http://localhost:8000/docs


INFO:     Started server process [3910]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


정상 케이스 테스트

`/docs` 페이지에서 `/chat` 엔드포인트를 열고 Try it out → Execute 순으로 호출해보세요.

요청 예시
```json
{
  "message": "안녕? 너는 누구야?"
}
```

응답 예시 (값은 실행마다 다름)
```json
{
  "reply": "안녕하세요! 저는 ...",
  "model": "openai/gpt-4o-mini"
}
```

## 7. 실패 케이스 살펴보기

`/docs` 또는 curl로 일부러 잘못된 요청을 보내보고, 어떤 응답이 오는지 살펴봅니다.

(1) 빈 메시지
- 요청: `{"message": ""}`
- 결과: 지금은 그대로 LLM에 전달되어 빈 응답이 오거나 어색한 답이 돌아옵니다.
- → TODO 3에서 400으로 막을 예정.

(2) 잘못된 타입
- 요청: `{"message": 123}`
- 결과: FastAPI/Pydantic이 자동으로 **422 Unprocessable Entity**를 반환합니다.
- Request Model 덕분에 우리가 따로 검증 코드를 짜지 않아도 막힌다는 점을 확인하세요.

(3) 잘못된 모델명 (옵션)
- `model_name = "이상한모델"` 처럼 일부러 잘못된 값으로 바꾸고 호출
- 결과: LLM 호출이 실패하면서 우리가 만든 try/except → `502 LLM 호출 실패` 가 떨어집니다.
- 확인 후 `model_name`은 원래 값으로 되돌려 두세요.

포인트
- **422**: Request Model이 입력 형식을 막아준다.
- **502**: LLM 호출 자체가 실패할 때 우리가 try/except로 막아 내려준다.
- **400**: "형식은 맞지만 의미상 비어 있는 입력"은 아직 안 막혀 있다 → TODO 3에서 처리.

세 가지 상태 코드가 각각 다른 역할을 한다는 점을 기억해두세요.

# TODO 실습

지금까지 만든 `/chat` 엔드포인트를 확장합니다. 각 TODO는 독립적으로 진행할 수 있지만, 순서대로 진행하면 자연스럽게 누적됩니다.

## TODO 1. Request Model 확장 - temperature 추가

- `ChatRequest`에 `temperature` 필드를 추가하세요.
- 조건
  - 선택 필드 (기본값 0.7)
  - 0.0 이상, 2.0 이하만 허용
- `/chat` 엔드포인트에서 `req.temperature`를 `call_llm`에 전달하도록 수정하세요.
- 확인: `/docs`에서 `temperature: 3.0` 으로 보내면 422가 떨어지는지 확인.

In [10]:
from pydantic import Field

class ChatRequest(BaseModel):
    message: str
    # TODO 1: temperature 필드 추가 (기본값 0.7, 0.0 ~ 2.0 범위)
    temperature: float = Field(0.7, ge=0.0, le=2.0)

app = FastAPI()

@app.post("/chat", response_model=ChatResponse)
def chat(req: ChatRequest):
    try:
        # TODO 1: call_llm 호출 시 temperature 전달
        response = call_llm(req.message, temperature=req.temperature)
    except Exception:
        raise HTTPException(status_code=502, detail="LLM 호출 실패")

    reply_text = response.choices[0].message.content
    return ChatResponse(reply=reply_text, model=response.model)

## TODO 2. Response Model 확장 - usage 추가

- `ChatResponse`에 `usage` 필드를 추가하세요.
- usage는 다음 정보를 담는 작은 모델로 정의합니다.
  - `prompt_tokens`: 입력 토큰 수
  - `completion_tokens`: 출력 토큰 수
  - `total_tokens`: 총 토큰 수
- `/chat` 엔드포인트에서 LLM 응답의 `usage`에서 값을 꺼내 채우세요.

In [12]:
class Usage(BaseModel):
    # TODO 2-1: 세 필드 모두 int 타입
    prompt_tokens: int
    completion_tokens: int
    total_tokens: int


class ChatResponse(BaseModel):
    reply: str
    model: str
    # TODO 2-2: usage 필드 추가 (타입: Usage)
    usage: Usage


app = FastAPI()


@app.post("/chat", response_model=ChatResponse)
def chat(req: ChatRequest):
    try:
        response = call_llm(req.message, temperature=req.temperature)
    except Exception:
        raise HTTPException(status_code=502, detail="LLM 호출 실패")

    reply_text = response.choices[0].message.content

    # TODO 2-3: response.usage 에서 값을 꺼내 Usage 인스턴스 생성
    usage = Usage(
        prompt_tokens=response.usage.prompt_tokens,
        completion_tokens=response.usage.completion_tokens,
        total_tokens=response.usage.total_tokens,
    )

    return ChatResponse(reply=reply_text, model=response.model, usage=usage)

## TODO 3. 입력 검증 강화 - 빈 메시지 차단

- 사용자가 `message`를 빈 문자열(`""`) 혹은 공백만 있는 문자열(`"   "`)로 보냈을 때
  - **400** 상태 코드로 응답하도록 처리하세요.
  - `detail` 메시지는 `"message는 비어 있을 수 없습니다."` 로 통일.
- 처리 위치는 `ChatRequest`의 validator 또는 `/chat` 함수 내부 중 자유롭게 선택.

In [14]:
app = FastAPI()


@app.post("/chat", response_model=ChatResponse)
def chat(req: ChatRequest):
    # TODO 3: req.message가 비어 있거나 공백만 있으면 400 에러 발생시키기
    # detail: "message는 비어 있을 수 없습니다."
    req.message = req.message.strip()
    if not req.message:
        raise HTTPException(status_code=400, detail="message는 비어 있을 수 없습니다.")


    try:
        response = call_llm(req.message, temperature=req.temperature)
    except Exception:
        raise HTTPException(status_code=502, detail="LLM 호출 실패")

    reply_text = response.choices[0].message.content
    usage = Usage(
        prompt_tokens=response.usage.prompt_tokens,
        completion_tokens=response.usage.completion_tokens,
        total_tokens=response.usage.total_tokens,
    )
    return ChatResponse(reply=reply_text, model=response.model, usage=usage)

## TODO 4. 오류 처리 분기

- LLM 호출 실패를 모두 502로만 내리지 말고, 오류 유형에 따라 다른 상태 코드로 내려주세요.
- 매핑
  - 인증 오류 → **401**
  - 한도 초과 → **429**
  - 그 외 외부 호출 실패 → **502**
- 힌트: `openai` 패키지의 예외 타입(`AuthenticationError`, `RateLimitError` 등)을 이용해 분기.
- `detail`에는 원인 카테고리만 짧게 표시 (예: `"LLM 호출 실패: rate limit"`)

In [15]:
# TODO 4
from openai import AuthenticationError, RateLimitError

app = FastAPI()

@app.post("/chat", response_model=ChatResponse)
def chat(req: ChatRequest):
    if not req.message.strip():
        raise HTTPException(status_code=400, detail="message는 비어 있을 수 없습니다.")

    # TODO 4: 오류 유형별로 분기 처리
    #   - AuthenticationError → 401, "LLM 호출 실패: authentication"
    #   - RateLimitError      → 429, "LLM 호출 실패: rate limit"
    #   - 그 외 Exception      → 502, "LLM 호출 실패"
    try:
        response = call_llm(req.message, temperature=req.temperature)
    except AuthenticationError:
        raise HTTPException(status_code=401, detail="LLM 호출 실패: authentication")
    except RateLimitError:
        raise HTTPException(status_code=429, detail="LLM 호출 실패: rate limit")
    except Exception:
        raise HTTPException(status_code=502, detail="LLM 호출 실패")

    reply_text = response.choices[0].message.content
    usage = Usage(
        prompt_tokens=response.usage.prompt_tokens,
        completion_tokens=response.usage.completion_tokens,
        total_tokens=response.usage.total_tokens,
    )
    return ChatResponse(reply=reply_text, model=response.model, usage=usage)

## TODO 5. /chat/v2 엔드포인트 추가

- 기존 `/chat`은 그대로 두고, 새로운 엔드포인트 `/chat/v2`를 추가하세요.
- 요구사항
  - 새 Request Model `ChatV2Request`
    - `system`: 시스템 프롬프트 (선택, 기본값 빈 문자열)
    - `message`: 사용자 메시지 (필수)
    - `temperature`: 0.0~2.0 (선택, 기본 0.7)
  - LLM 호출 시 `system`이 비어있지 않으면 `messages`에 `system` role 메시지를 먼저 추가.
  - 응답은 `ChatResponse`(또는 확장한 모델) 그대로 사용 가능.
- 확인: `/docs`에서 `system="너는 친절한 도우미야"` 와 빈 system 두 경우의 답변 톤 차이를 비교.

In [16]:
# TODO 5-1: ChatV2Request 정의
class ChatV2Request(BaseModel):
    system: str = ""   # 선택, 기본값 빈 문자열
    message: str       # 필수
    temperature: float = Field(0.7, ge=0.0, le=2.0)  # Field로 0.0~2.0, 기본 0.7


def call_llm_v2(system: str, message: str, temperature: float):
    messages = []

    # TODO 5-2: system 이 비어있지 않을 때만 system 메시지 추가, 그 다음 user 메시지 추가
    if system.strip():
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": message})

    return client.chat.completions.create(
        model=model_name,
        messages=messages,
        temperature=temperature,
    )


@app.post("/chat/v2", response_model=ChatResponse)
def chat_v2(req: ChatV2Request):
    # 5-3: 빈 메세지면 400
    req.message = req.message.strip()
    if not req.message:
        raise HTTPException(status_code=400, detail="message는 비어 있을 수 없습니다.")

    # 5-4: call_llm_v2 호출 + 예외 분기
    #   - AuthenticationError → 401, "LLM 호출 실패: authentication"
    #   - RateLimitError      → 429, "LLM 호출 실패: rate limit"
    #   - 그 외 Exception      → 502, "LLM 호출 실패"
    try:
        response = call_llm_v2(req.system, req.message, req.temperature)
    except AuthenticationError:
        raise HTTPException(status_code=401, detail="LLM 호출 실패: authentication")
    except RateLimitError:
        raise HTTPException(status_code=429, detail="LLM 호출 실패: rate limit")
    except Exception:
        raise HTTPException(status_code=502, detail="LLM 호출 실패")
    

    # 5-5: 응답 조립 - reply-text, usage(usage), ChatResponse 반환
    reply_text = response.choices[0].message.content
    usage = Usage(
        prompt_tokens=response.usage.prompt_tokens,
        completion_tokens=response.usage.completion_tokens,
        total_tokens=response.usage.total_tokens,
    )


    return ChatResponse(reply=reply_text, model=response.model, usage=usage)

## 최종 확인 - 서버 다시 띄우기

TODO를 모두 반영한 `app`을 새로운 포트에서 다시 띄워 동작을 확인해보세요.

- 이전 서버가 살아있으면 `server.should_exit = True`로 먼저 멈추거나 포트 번호를 바꿔서 띄웁니다.
- `/docs`에서 `/chat`, `/chat/v2`가 모두 정상 동작하는지 확인하세요.
- 빈 메시지, temperature 범위 밖 값, `system` 프롬프트 있는/없는 경우를 모두 호출해 응답 차이를 비교해보세요.

In [ ]:
config = Config(app=app, host="0.0.0.0", port=8001, log_level="info")
server = Server(config)

thread = threading.Thread(target=server.run, daemon=True)
thread.start()

print("서버 실행 중: http://localhost:8001")
print("문서 페이지: http://localhost:8001/docs")

서버 실행 중: http://localhost:8001
문서 페이지: http://localhost:8001/docs


INFO:     Started server process [3910]
INFO:     Waiting for application startup.
INFO:     Application startup complete.


INFO:     Uvicorn running on http://0.0.0.0:8001 (Press CTRL+C to quit)


## 실습 정리

- 하나의 FastAPI 앱에서 `/chat`, `/chat/v2`가 동작
- 입력 검증 → LLM 호출 → 오류 분기 → 응답 정리까지 전체 파이프라인 완성
- 이 형태가 실제 LLM 백엔드에서 가장 기본이 되는 구조입니다.